<a href="https://colab.research.google.com/github/kazumah1/single-word-decoding/blob/multi-timescale-CNN/colab_trainer_memory_efficient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MEG Sentence Decoding — `multi-timescale-CNN` branch

**One-time setup (do this before running):**
1. Open the shared Drive folder: https://drive.google.com/drive/folders/1HOvzJEv0Czn-yJKELq6kp5cORJ5yBTdM
2. Right-click it → **Organize** → **Add shortcut to Drive** → place it in **My Drive** and name it exactly `single-word-decoding`

**Workflow**

1. Run intial setup (Section 1, steps 1.1-1.5*)
2. Tune Hyperparams (Optional - Section 2, steps 2.1, 2.2, 2.4)
3. Train model (section 3)
4. View results (section 4)


**Data & results strategy:**
- All branches share the same Drive data folder (no re-downloading across branches)
- Checkpoints and results for this branch go to `single-word-decoding/results/multi-timescale-CNN/`


## Section 1: Setup

---


This section will import the necessary datafiles and dependecies and build the repository for training.

### Step 1.1: Mount drive

Run this first. It will ask you to sign in and authorize access.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login
import os, sys, shutil, torch
drive.mount('/content/drive')

# Auth HuggingFace for llama
token = userdata.get('HF_TOKEN')
login(token)


# ── Branch ─────────────────────────────────────────────────────────────
BRANCH         = 'multi-timescale-CNN'

# ── Paths ──────────────────────────────────────────────────────────────
TRAIN_SUBJECTS = ['sub-01','sub-02','sub-03','sub-04','sub-05','sub-06','sub-07','sub-08','sub-09','sub-10',
                  'sub-11','sub-12','sub-13','sub-14','sub-15','sub-16','sub-17','sub-18','sub-19','sub-20',
                  'sub-21','sub-22','sub-23','sub-24','sub-25','sub-26','sub-27']

REPO           = 'https://github.com/kazumah1/single-word-decoding.git'
WORKDIR        = f'/content/single-word-decoding-{BRANCH.replace("/", "-")}'
LOCAL_DATAPATH = '/content/neural_data'

# Where your data already lives on Drive
DRIVE_GW       = '/content/drive/MyDrive/datasets/gwilliams2022/download'

# The final destination for results on Google Drive
DRIVE_SAVEPATH = f'/content/drive/MyDrive/sentence-results/{BRANCH.replace("/", "-")}'
# A temporary local path for all intermediate training outputs (checkpoints, logs, etc.)
SAVEPATH       = f'/content/tmp_training_output/{BRANCH.replace("/", "-")}'
# ───────────────────────────────────────────────────────────────────────

os.makedirs(f'{DRIVE_SAVEPATH}/cache', exist_ok=True) # Create cache dir on Drive if needed for *final* results
os.makedirs(SAVEPATH, exist_ok=True) # Create local temporary path for current run
os.makedirs(f'{SAVEPATH}/cache', exist_ok=True) # Create local temporary cache dir
os.makedirs(LOCAL_DATAPATH, exist_ok=True)

print(f"Branch: {BRANCH}")
print(f"GPU:    {torch.cuda.get_device_name(0)}")
total, _, free = shutil.disk_usage('/content')
print(f"Disk:   {free/1e9:.0f} GB free / {total/1e9:.0f} GB total")
print(f"\nTrain subjects: {TRAIN_SUBJECTS}")
print(f"Local training results path: {SAVEPATH}")
print(f"Final Google Drive results path: {DRIVE_SAVEPATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Branch: multi-timescale-CNN
GPU:    NVIDIA A100-SXM4-80GB
Disk:   43 GB free / 253 GB total

Train subjects: ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27']
Local training results path: /content/tmp_training_output/multi-timescale-CNN
Final Google Drive results path: /content/drive/MyDrive/sentence-results/multi-timescale-CNN


### Step 1.2: Instance Setup (run once per session, then walk away)

This cell clones the `multi-timescale-CNN` branch, installs all dependencies, and transfers the data into the current runtime for training:

In [ ]:
import os, shutil, subprocess, sys

# ── 1. Clone repo ──────────────────────────────────────────────────────
print(f'=== Cloning branch: {BRANCH} ===')
if os.path.exists(WORKDIR):
    print('  Already cloned, pulling latest...')
    subprocess.run(['git', '-C', WORKDIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
if not os.path.exists(f'{WORKDIR}/sentence_results'):
    os.symlink(SAVEPATH, f'{WORKDIR}/sentence_results')
os.makedirs(f'{WORKDIR}/projects', exist_ok=True)
print('  Done.')

# ── 2. Install local packages ──────────────────────────────────────────
print('\n=== Installing local packages ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuralset/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--config-settings', 'editable_mode=strict', '-e', 'neuraltrain/'], check=True)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
print('  Done.')

# ── 3. Install dependencies ────────────────────────────────────────────
print('\n=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightning', 'pytorch-lightning', 'torchvision',
                'wandb', 'osfclient', 'mne_bids', 'tqdm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'x-transformers==1.26.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics==1.5.2'], check=True)
print('  Done.')

# ── 4. kenlm ──────────────────────────────────────────────────────────
print('\n=== Installing kenlm ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kenlm'], capture_output=True)
try:
    import kenlm; print('  Installed from PyPI.')
except ImportError:
    print('  PyPI failed — building from source...')
    subprocess.run(['git', 'clone', 'https://github.com/kpu/kenlm', '/content/kenlm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cython'], check=True)
    subprocess.run(['cython', '/content/kenlm/python/kenlm.pyx', '--cplus'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '/content/kenlm'], check=True)
    import kenlm; print('  Built from source.')

# ── 5. Copy training subjects from Drive to local runtime ─────────────────
print('\n=== Copying data from Drive to local disk ===')
local_gw = f'{LOCAL_DATAPATH}/gwilliams2022/download'
os.makedirs(local_gw, exist_ok=True)

for item in ['stimuli'] + TRAIN_SUBJECTS:
    src = f'{DRIVE_GW}/{item}'
    dst = f'{local_gw}/{item}'
    if os.path.exists(dst):
        print(f'  {item} already on local disk, skipping.')
    elif os.path.exists(src):
        print(f'  Copying {item}...')
        shutil.copytree(src, dst)
    else:
        print(f'  WARNING: {item} not found at {src}')

_, _, free = shutil.disk_usage('/content')
print(f'  Disk free: {free/1e9:.0f} GB')
print(f'\n✓ Setup complete. Run Cell 3 to train on branch: {BRANCH}')

=== Cloning branch: multi-timescale-CNN ===
  Already cloned, pulling latest...
  Done.

=== Installing local packages ===
  Done.

=== Installing dependencies ===
  Done.

=== Installing kenlm ===
  Installed from PyPI.

=== Copying data from Drive to local disk ===
  stimuli already on local disk, skipping.
  sub-01 already on local disk, skipping.
  sub-02 already on local disk, skipping.
  sub-03 already on local disk, skipping.
  sub-04 already on local disk, skipping.
  sub-05 already on local disk, skipping.
  sub-06 already on local disk, skipping.
  sub-07 already on local disk, skipping.
  sub-08 already on local disk, skipping.
  sub-09 already on local disk, skipping.
  sub-10 already on local disk, skipping.
  sub-11 already on local disk, skipping.
  sub-12 already on local disk, skipping.
  sub-13 already on local disk, skipping.
  sub-14 already on local disk, skipping.
  sub-15 already on local disk, skipping.
  sub-16 already on local disk, skipping.
  sub-17 already 

### Step 1.3: Extract Relevant Classes for Training

Run the following cell to extract the relevant ```multiscale-``` variants of the ```simpleconv``` models for training:

In [ ]:
import re

path = f'{WORKDIR}/neuraltrain/neuraltrain/models/multiscaleconv.py'
with open(path) as f:
    src = f.read()

# 1. Add imports from simpleconv after the transformer import line
src = src.replace(
    'from .transformer import LlamaTransformerConfig, TransformerEncoderConfig',
    'from .transformer import LlamaTransformerConfig, TransformerEncoderConfig\n'
    'from .simpleconv import (\n'
    '    ConvSequence, SpatialFilter,\n'
    '    SimpleConvConfig, SimpleConv,\n'
    '    SimpleConvTimeAggConfig, SimpleConvTimeAgg,\n'
    ')'
)

# 2. Remove the duplicate class blocks for SimpleConvConfig, SimpleConv,
#    SimpleConvTimeAggConfig, SimpleConvTimeAgg — everything between
#    MultiScaleConvSequence and MultiScaleSimpleConvConfig
src = re.sub(
    r'(\n# -{5,}\n# MultiScaleConvSequence.*?)\n# -{5,}\n# SimpleConvConfig.*?\n# -{5,}\n# MultiScaleSimpleConvConfig',
    r'\1\n\n# ---------------------------------------------------------------------------\n# MultiScaleSimpleConvConfig',
    src, flags=re.DOTALL
)

# 3. Remove duplicate SimpleConvTimeAggConfig and SimpleConvTimeAgg blocks
#    (between MultiScaleSimpleConv and MultiScaleSimpleConvTimeAgg)
src = re.sub(
    r'(\nclass MultiScaleSimpleConv\b.*?)\nclass SimpleConvTimeAggConfig\b.*?\nclass MultiScaleSimpleConvTimeAggConfig',
    r'\1\n\nclass MultiScaleSimpleConvTimeAggConfig',
    src, flags=re.DOTALL
)

with open(path, 'w') as f:
    f.write(src)

# Verify
classes = [l for l in src.splitlines() if l.startswith('class ')]
print('Classes in file:')
for c in classes:
    print(' ', c)


Classes in file:
  class SpatialFilter(nn.Module):
  class ConvSequence(nn.Module):
  class MultiScaleConvSequence(nn.Module):
  class MultiScaleSimpleConvConfig(SimpleConvConfig):
  class SimpleConv(nn.Module):
  class MultiScaleSimpleConv(SimpleConv):
  class MultiScaleSimpleConvTimeAggConfig(MultiScaleSimpleConvConfig):
  class MultiScaleSimpleConvTimeAgg(MultiScaleSimpleConv):


### Step 1.4: Extract Training Config from ```sentence_decoding/grids/```

Run the following cell to build the experiment for training:

In [ ]:
path = f'{WORKDIR}/sentence_decoding/grids/test.py'
with open(path) as f:
    src = f.read()

# Replace the current n_timelines value with 10000
src = src.replace('"data.n_timelines": 999999999999999999', '"data.n_timelines": 10000')
src = src.replace('"save_checkpoints": True', '"save_checkpoints": False') # Set to False to avoid saving checkpoints

with open(path, 'w') as f:
    f.write(src)
print('n_timelines:', '10000' if '"data.n_timelines": 10000' in src else 'FAILED')
print('save_checkpoints:', 'False' if '"save_checkpoints": False' in src else 'FAILED')

n_timelines: 10000
save_checkpoints: False


### Step 1.5: Delete Existing Study Info (Optional)

Run this cell is you are restarting/starting an experiment to delete existing cache data:

In [ ]:
import shutil
path = f'{SAVEPATH}/cache/sentence_decoding/StudyLoader,0-v2,Gwilliams2022/neuralset.data.StudyLoader._build,ntimelines=10000,name=Gwilliams2022-9f9397f9'
shutil.rmtree(path)
print('Deleted')


## Section 2: Hyperparameter Tuning

---

The following section implements bayesian hyperparameter tuning, an advanced hyperparameter tuning strategy that uses Baye's rule from conditional probability to search for optimal hyperparameters to test next. This, in theory, allows for larger, more expansive search spaces than traditional grid searches by allowing for informed automated grid selection.

### Step 2.1: Tuning Setup

Make sure to edit and run the following cell to install necessary dependencies and define your search bounds:

In [ ]:
# ============================================================
# Bayesian HPO — Cell A: install dependencies & define search
# ============================================================
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'optuna>=3.6', 'plotly>=5.20', 'kaleido==0.2.1'], check=True)

# Env vars must be set BEFORE importing sentence_decoding (defaults.py reads them at import time)
os.environ['DATAPATH']   = LOCAL_DATAPATH
os.environ['SAVEPATH']   = SAVEPATH
os.environ['WANDB_MODE'] = 'disabled'
os.environ['PYTHONPATH'] = WORKDIR
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

# ---- Search space ------------------------------------------------------
# Each entry maps an Optuna param name to (a) a distribution and
# (b) the flattened config key it maps onto inside default_config.
SEARCH_SPACE = {
    'lr':                {'type': 'loguniform', 'low': 1e-5, 'high': 5e-3,
                          'cfg': 'trainer_config.lr'},
    'weight_decay':      {'type': 'loguniform', 'low': 1e-7, 'high': 1e-2,
                          'cfg': 'trainer_config.weight_decay'},
    'gradient_clip_val': {'type': 'uniform',    'low': 0.0,  'high': 5.0,
                          'cfg': 'trainer_config.gradient_clip_val'},
    'dropout_input':     {'type': 'uniform',    'low': 0.0,  'high': 0.5,
                          'cfg': 'brain_model_config.dropout_input'},
    'hidden':            {'type': 'categorical','choices': [128, 160, 256, 320],
                          'cfg': 'brain_model_config.hidden'},
    'depth':             {'type': 'int',        'low': 3, 'high': 7,
                          'cfg': 'brain_model_config.depth'},
    'kernel_size':       {'type': 'categorical','choices': [3, 5, 7],
                          'cfg': 'brain_model_config.kernel_size'},
    'dilation_period':   {'type': 'int',        'low': 3, 'high': 7,
                          'cfg': 'brain_model_config.dilation_period'},
    'glu':               {'type': 'int',        'low': 1, 'high': 3,
                          'cfg': 'brain_model_config.glu'},
    'initial_linear':    {'type': 'categorical','choices': [256, 512, 768],
                          'cfg': 'brain_model_config.initial_linear'},
    'spatial_filters':   {'type': 'categorical','choices': [16, 32, 64],
                          'cfg': 'brain_model_config.spatial_filters'},
    'batch_size':        {'type': 'categorical','choices': [64, 128, 256],
                          'cfg': 'data.batch_size'},
}

# ---- HPO configuration -------------------------------------------------
# Each trial is aggressively down-scaled so the 8B-param ensemble is tractable.
# - use_transformer=False skips the frozen Llama decoder (the dominant cost);
#   we're only tuning the CNN encoder + optimizer, which is the under-trained part.
# - n_timelines/n_subjects use small, cache-stable subsets.
# - max_epochs_per_trial is short; pruner kills losers even sooner.
HPO_CONFIG = {
    'n_trials':              25,     # total Bayesian trials (raise for more coverage)
    'max_epochs_per_trial':  5,      # early-stop ceiling per trial
    'n_timelines_per_trial': 800,    # data subset (caches after first trial)
    'n_subjects_per_trial':  6,      # subject subset (caches after first trial)
    'use_transformer_in_search': False,
    'proxy_metric':          'val_contrastive_top5_acc',  # higher = better
    'proxy_mode':            'max',
    'pruner_warmup_epochs':  2,
    'pruner_min_trials':     4,
    'study_name':            'meg_hpo_v1',
    'storage_dir':           f'{SAVEPATH}/hpo',
    'seed':                  0,
}
os.makedirs(HPO_CONFIG['storage_dir'], exist_ok=True)

print(f"Tunable HPs ({len(SEARCH_SPACE)}): {list(SEARCH_SPACE.keys())}")
print(f"Trials:  {HPO_CONFIG['n_trials']}  "
      f"Epochs/trial: {HPO_CONFIG['max_epochs_per_trial']}  "
      f"Subjects/trial: {HPO_CONFIG['n_subjects_per_trial']}  "
      f"Timelines/trial: {HPO_CONFIG['n_timelines_per_trial']}")
print(f"Proxy metric: {HPO_CONFIG['proxy_metric']} ({HPO_CONFIG['proxy_mode']})")
print(f"HPO artifacts: {HPO_CONFIG['storage_dir']}")


### Step 2.2: Search

Run the following cell to perform Bayesian HPO based on the config outlined in step 1:

In [ ]:
# ============================================================
# Bayesian HPO — Cell B: TPE + MedianPruner optimization loop
# ============================================================
import os, gc, time, json, copy, warnings, contextlib
import torch
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import lightning.pytorch as pl
from lightning.pytorch.callbacks import Callback, LearningRateMonitor, EarlyStopping

# Import AFTER env vars were set in Cell A so cache/save dirs resolve correctly
from sentence_decoding.main import Experiment
from sentence_decoding.callbacks import ShuffleSentences
from sentence_decoding.grids.defaults import default_config
from neuraltrain.utils import update_config

warnings.filterwarnings('ignore', category=UserWarning)

MONITOR = HPO_CONFIG['proxy_metric']
MODE    = HPO_CONFIG['proxy_mode']

# ---- Lightning callback that pipes val metric to Optuna each epoch -----
class OptunaPruneCallback(Callback):
    def __init__(self, trial, monitor):
        self.trial, self.monitor = trial, monitor
    def on_validation_epoch_end(self, trainer, pl_module):
        cm = trainer.callback_metrics
        val = cm.get(self.monitor, None)
        if val is None:  # Lightning may append a dataloader suffix
            hits = [v for k, v in cm.items() if k.startswith(self.monitor)]
            val = hits[0] if hits else None
        if val is None:
            return
        self.trial.report(float(val), step=trainer.current_epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()

def _suggest(trial, name, spec):
    t = spec['type']
    if t == 'loguniform':  return trial.suggest_float(name, spec['low'], spec['high'], log=True)
    if t == 'uniform':     return trial.suggest_float(name, spec['low'], spec['high'])
    if t == 'int':         return trial.suggest_int(name, spec['low'], spec['high'])
    if t == 'categorical': return trial.suggest_categorical(name, spec['choices'])
    raise ValueError(t)

def _read_monitor(trainer):
    cm = trainer.callback_metrics
    if MONITOR in cm: return float(cm[MONITOR])
    hits = [v for k, v in cm.items() if k.startswith(MONITOR)]
    return float(hits[0]) if hits else (float('-inf') if MODE == 'max' else float('inf'))

# ---- Build a per-trial config ------------------------------------------
def build_trial_config(trial):
    sample = {n: _suggest(trial, n, s) for n, s in SEARCH_SPACE.items()}
    params = {
        'seed': HPO_CONFIG['seed'],
        'infra.cluster': None,
        'use_wandb': False,
        'save_checkpoints': False,
        'use_transformer': HPO_CONFIG['use_transformer_in_search'],
        'data.n_timelines':  HPO_CONFIG['n_timelines_per_trial'],
        'data.n_subjects':   HPO_CONFIG['n_subjects_per_trial'],
        'data.num_workers':  4,
        'trainer_config.n_epochs':  HPO_CONFIG['max_epochs_per_trial'],
        'trainer_config.patience':  max(2, HPO_CONFIG['max_epochs_per_trial'] // 2),
        'trainer_config.monitor':   MONITOR,
        'trainer_config.transformer_start_epoch': 10_000,  # never activate during search
    }
    # Map optuna-suggested values into nested config keys
    for name, val in sample.items():
        params[SEARCH_SPACE[name]['cfg']] = val
    cfg = update_config(default_config, params)
    cfg['infra']['folder'] = os.path.join(
        HPO_CONFIG['storage_dir'], 'trials', f'trial_{trial.number:03d}')
    os.makedirs(cfg['infra']['folder'], exist_ok=True)
    return cfg, sample

# ---- Objective ---------------------------------------------------------
def objective(trial):
    torch.cuda.empty_cache(); gc.collect()
    cfg, sample = build_trial_config(trial)
    t0 = time.time()

    task = Experiment(**cfg)
    task.infra.clear_job()
    loaders = task.setup_run()

    brain_model, transformer = task.get_model(loaders['train'])
    module = task.load_module(brain_model, transformer)

    callbacks = [
        LearningRateMonitor(logging_interval='epoch'),
        ShuffleSentences(),
        EarlyStopping(monitor=MONITOR,
                      patience=task.trainer_config.patience,
                      mode=MODE, verbose=False, check_finite=False),
        OptunaPruneCallback(trial, MONITOR),
    ]
    trainer = pl.Trainer(
        gradient_clip_val=task.trainer_config.gradient_clip_val,
        devices=task.infra.gpus_per_node,
        max_epochs=task.trainer_config.n_epochs,
        enable_progress_bar=False,
        log_every_n_steps=50,
        logger=False,
        callbacks=callbacks,
        inference_mode=False,
        enable_checkpointing=False,
        num_sanity_val_steps=0,
    )
    pl.seed_everything(task.seed, verbose=False)
    try:
        trainer.fit(module,
                    train_dataloaders=loaders['train'],
                    val_dataloaders=loaders['val'])
        score = _read_monitor(trainer)
    finally:
        del task, brain_model, transformer, module, trainer, loaders
        torch.cuda.empty_cache(); gc.collect()

    print(f"[trial {trial.number:03d}] {MONITOR}={score:.4f}  "
          f"({time.time()-t0:.0f}s)  params={sample}")
    return score

# ---- Study -------------------------------------------------------------
sampler = TPESampler(
    seed=HPO_CONFIG['seed'], multivariate=True, group=True,
    constant_liar=True, n_startup_trials=5,
)
pruner = MedianPruner(
    n_startup_trials=HPO_CONFIG['pruner_min_trials'],
    n_warmup_steps=HPO_CONFIG['pruner_warmup_epochs'],
)
storage = f"sqlite:///{HPO_CONFIG['storage_dir']}/{HPO_CONFIG['study_name']}.db"
study = optuna.create_study(
    study_name=HPO_CONFIG['study_name'],
    direction='maximize' if MODE == 'max' else 'minimize',
    storage=storage, load_if_exists=True,
    sampler=sampler, pruner=pruner,
)

print(f"\n{'='*60}\nStarting {HPO_CONFIG['n_trials']} Bayesian trials\n{'='*60}")
t_start = time.time()
study.optimize(
    objective,
    n_trials=HPO_CONFIG['n_trials'],
    gc_after_trial=True,
    show_progress_bar=False,
    catch=(RuntimeError,),   # don't kill the search on a single OOM
)
elapsed = (time.time() - t_start) / 60.0

print(f"\n{'='*60}\nDone in {elapsed:.1f} min.  "
      f"Best trial: #{study.best_trial.number}  "
      f"Best {MONITOR}: {study.best_value:.4f}\n{'='*60}")
for k, v in study.best_params.items():
    print(f"  {k:<20s} = {v}")


### Step 2.3: Analyze Search Results (Optional)

The following cell visualizes the results of the Bayesian search done above for evaluation purposes:

In [ ]:
# ============================================================
# Bayesian HPO — Cell C: evaluation figures
# ============================================================
import os, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import optuna.visualization as vis

fig_dir = os.path.join(HPO_CONFIG['storage_dir'], 'figures')
os.makedirs(fig_dir, exist_ok=True)

# ---- Interactive Optuna figures (Plotly, render inline + save PNG) -----
for title, fn in [
    ('optimization_history',  vis.plot_optimization_history),
    ('parameter_importances', vis.plot_param_importances),
    ('parallel_coordinate',   vis.plot_parallel_coordinate),
    ('slice',                 vis.plot_slice),
    ('intermediate_values',   vis.plot_intermediate_values),
    ('edf',                   vis.plot_edf),
]:
    try:
        fig = fn(study)
        fig.update_layout(title=title.replace('_', ' ').title())
        display(fig)
        fig.write_image(os.path.join(fig_dir, f'{title}.png'),
                        width=1000, height=600, scale=2)
    except Exception as e:
        print(f'  skipped {title}: {e}')

# ---- Summary table of completed trials ---------------------------------
df = study.trials_dataframe(attrs=('number','value','state','duration','params'))
df = df.sort_values('value', ascending=(HPO_CONFIG['proxy_mode'] != 'max'))
df.to_csv(os.path.join(HPO_CONFIG['storage_dir'], 'trials.csv'), index=False)
print(f'\nTop 10 trials by {HPO_CONFIG["proxy_metric"]}:')
display(df.head(10))

# ---- Static matplotlib summary (always renders even without kaleido) ---
completed = [t for t in study.trials if t.value is not None]
pruned    = [t for t in study.trials if t.state.name == 'PRUNED']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# (1) Running best score over trials
vals = [t.value for t in completed]
running_best = []
best = -float('inf') if HPO_CONFIG['proxy_mode'] == 'max' else float('inf')
for v in vals:
    best = max(best, v) if HPO_CONFIG['proxy_mode'] == 'max' else min(best, v)
    running_best.append(best)
axes[0].plot(range(len(vals)), vals, 'o', alpha=0.5, label='trial score')
axes[0].plot(range(len(running_best)), running_best, '-', lw=2, label='running best')
axes[0].set_xlabel('completed trial'); axes[0].set_ylabel(HPO_CONFIG['proxy_metric'])
axes[0].set_title(f'Optimization progress  ({len(pruned)} pruned)'); axes[0].legend()
axes[0].grid(alpha=0.3)

# (2) Per-hyperparameter scatter of score vs value (visual sensitivity)
if completed:
    best_params = study.best_params
    # pick the 4 params with highest importance (fallback to first 4)
    try:
        imp = optuna.importance.get_param_importances(study)
        top_params = list(imp.keys())[:4]
    except Exception:
        top_params = list(best_params.keys())[:4]
    ax = axes[1]
    for p in top_params:
        xs, ys = [], []
        for t in completed:
            if p in t.params:
                xs.append(t.params[p]); ys.append(t.value)
        ax.scatter(range(len(xs)), ys, label=p, alpha=0.7)
    ax.set_xlabel('trial order'); ax.set_ylabel(HPO_CONFIG['proxy_metric'])
    ax.set_title('Top-importance HPs vs score'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Persist best params ----------------------------------------------
with open(os.path.join(HPO_CONFIG['storage_dir'], 'best_params.json'), 'w') as f:
    json.dump({
        'metric':      HPO_CONFIG['proxy_metric'],
        'mode':        HPO_CONFIG['proxy_mode'],
        'best_value':  study.best_value,
        'best_trial':  study.best_trial.number,
        'best_params': study.best_params,
        'n_trials':    len(study.trials),
        'n_pruned':    len(pruned),
    }, f, indent=2)
print(f'\nArtifacts saved to: {HPO_CONFIG["storage_dir"]}')


### Step 2.4: Update Config

Run the following cell to update ```test.py``` to use the optimal params found by bayesian HPO:

In [ ]:
# ============================================================
# Bayesian HPO — Cell D: inject best HPs into the training config
# ============================================================
import re, json, os

best_params = study.best_params
cfg_updates = {SEARCH_SPACE[k]['cfg']: v for k, v in best_params.items()}

test_path = f'{WORKDIR}/sentence_decoding/grids/test.py'
with open(test_path) as f:
    src = f.read()

inject = ',\n    '.join([f'"{k}": {repr(v)}' for k, v in cfg_updates.items()])
block = (
    "\n# --- Auto-generated from Bayesian HPO (do not edit by hand) ---\n"
    f"_bayes_best = {{\n    {inject},\n}}\n"
    "default = update_config(default, _bayes_best)\n"
    "# --- end auto-generated ---\n"
)

if '# --- Auto-generated from Bayesian HPO' in src:
    src = re.sub(
        r'\n# --- Auto-generated from Bayesian HPO.*?# --- end auto-generated ---\n',
        block, src, flags=re.DOTALL,
    )
else:
    src = src.replace(
        'default = update_config(default_config, default_params)',
        'default = update_config(default_config, default_params)' + block,
    )

with open(test_path, 'w') as f:
    f.write(src)

print(f'Patched {test_path} with:')
print(json.dumps(cfg_updates, indent=2))
print('\nNow re-run Cell 3 — training will use the best HPs with the full 8B ensemble.')


## Section 3: Model Training

---

Use the following cell to train the model outlined by current state of ```setence_decoding/grids/test.py```:

In [ ]:
import os
import shutil
import glob

# Set environment variables for the training script
os.environ['DATAPATH']   = LOCAL_DATAPATH
os.environ['SAVEPATH']   = SAVEPATH # SAVEPATH now points to the local temporary path
os.environ['WANDB_MODE'] = 'disabled'  # set to 'online' to enable wandb if needed
os.environ['PYTHONPATH'] = WORKDIR

print(f"Running training with local SAVEPATH: {SAVEPATH}")

# Debug: Check if the cache directory exists
expected_cache_dir = f'{SAVEPATH}/cache'
print(f"Checking existence of {expected_cache_dir}: {os.path.exists(expected_cache_dir)}")

# Run the training script
!DATAPATH={LOCAL_DATAPATH} SAVEPATH={SAVEPATH} WANDB_MODE={os.environ['WANDB_MODE']} PYTHONPATH={WORKDIR} \
    python -m sentence_decoding.grids.test

# After training, copy final results to Google Drive
print("\nCopying final results to Google Drive...")

# Identify the actual output directory generated by the training script within the local SAVEPATH
# This assumes a structure like SAVEPATH/results/sentence_decoding/<timestamp>
local_result_base_path = f'{SAVEPATH}/results/sentence_decoding'
drive_result_base_path = f'{DRIVE_SAVEPATH}/results/sentence_decoding' # Use DRIVE_SAVEPATH for final destination

# Ensure the destination directory on Drive exists
os.makedirs(drive_result_base_path, exist_ok=True)

latest_local_run_dir = None
if os.path.exists(local_result_base_path):
    # Find the latest created directory within the local results path
    all_local_runs = sorted(glob.glob(f'{local_result_base_path}/*'))
    if all_local_runs:
        latest_local_run_dir = all_local_runs[-1]

if latest_local_run_dir and os.path.isdir(latest_local_run_dir):
    print(f"Found latest local run directory: {latest_local_run_dir}")
    # Construct the destination path on Drive
    dest_dir_name = os.path.basename(latest_local_run_dir)
    final_drive_dest_path = os.path.join(drive_result_base_path, dest_dir_name)

    # Copy the entire directory (containing metrics.json etc.) to Drive
    print(f"Copying {latest_local_run_dir} to {final_drive_dest_path}...")
    shutil.copytree(latest_local_run_dir, final_drive_dest_path)
    print("Copy complete.")

    # Clean up local temporary directory
    print(f"Cleaning up local temporary directory: {SAVEPATH}")
    shutil.rmtree(SAVEPATH)
    print("Local temporary directory cleared.")
else:
    print("No training results found in the local temporary path to copy.")

Running training with local SAVEPATH: /content/tmp_training_output/multi-timescale-CNN
Checking existence of /content/tmp_training_output/multi-timescale-CNN/cache: True
2026-04-17 09:30:09 - WARNING - neuralset.infra.utils:212 - Did not find a discriminator for transformer_config (uid may be incomplete)
2026-04-17 09:30:09 - INFO - exca.map:500 - Sent 16 items for StudyLoader,0-v2,Gwilliams2022/neuralset.data.StudyLoader._load_timelines,ntimelines=10000,name=Gwilliams2022-9f9397f9 into a processpool
  0% 0/16 [00:00<?, ?it/s]Extracting SQD Parameters from /content/neural_data/gwilliams2022/download/sub-07/ses-1/meg/sub-07_ses-1_task-1_meg.con...
Creating Raw.info structure...
Extracting SQD Parameters from /content/neural_data/gwilliams2022/download/sub-07/ses-1/meg/sub-07_ses-1_task-2_meg.con...
Creating Raw.info structure...
Extracting SQD Parameters from /content/neural_data/gwilliams2022/download/sub-07/ses-0/meg/sub-07_ses-0_task-1_meg.con...
Extracting SQD Parameters from /conte

## Section 4: Results

---

The following cells extract and save the results of the experiment ran within section 3:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import subprocess
result = subprocess.run(['find', SAVEPATH, '-name', '*.ckpt'], capture_output=True, text=True)
print(result.stdout)

KeyboardInterrupt: 

In [ ]:
import glob, json, os

result_dirs = sorted(glob.glob(f'{SAVEPATH}/results/sentence_decoding/*'))
if not result_dirs:
    # Fallback: search one level up
    result_dirs = sorted(glob.glob(f'{SAVEPATH}/**/*', recursive=False))
if not result_dirs:
    print('No results yet.')
else:
    latest = result_dirs[-1]
    print(f'Branch:     {BRANCH}')
    print(f'Latest run: {latest}')
    for f in sorted(os.listdir(latest)):
        print(f'  {f}')

    metrics_path = os.path.join(latest, 'metrics.json')
    if os.path.exists(metrics_path):
        with open(metrics_path) as fh:
            metrics = json.load(fh)
        print('\nMetrics:')
        for k, v in metrics.items():
            print(f'  {k}: {v}')


In [ ]:
print(WORKDIR)


/content/single-word-decoding-multi-timescale-CNN
